In [1]:
## Uncomment and run the below to commands for reloading the modules of un_llm_analysis after changes. 
# %load_ext autoreload
# %autoreload 2

# Identifying Issues for Analysis of UN Briefings

This notebook demonstrates how the issues used in the subsequent analysis are identified. It analyzed the section titles from the past 5 years of UN daily briefing data, focusing on the most frequently recurring titles that represent substantive issues.

## 1. Set up and load data

This notebook uses the pre-uploaded database specified by `BRIEFINGS_DB_PATH`. 

Alternatively, the database can be created from scratch for a desired date range:
```bash
briefings update 2026-09-01 2026-09-10
briefings init
briefings load 2026-09-01 2026-09-10
```


In [2]:
import pandas as pd
import ollama
from tqdm import tqdm

from un_llm_analysis.config import BASE_URL, BRIEFINGS_DB_PATH
from un_llm_analysis.database import (
    get_db_connection, 
    get_briefing_id_from_date, 
    get_briefings_between_dates,
)

In [3]:
# Get the data in the database with its maximum date range
db_start_date = "2021-01-01"
db_end_date = "2026-09-10"

with get_db_connection(BRIEFINGS_DB_PATH) as conn:
    briefings = get_briefings_between_dates(conn, db_start_date, db_end_date)

In [4]:
# Get the section data from the briefings and store it in a DataFrame

rows = []

for briefing in tqdm(briefings):
    briefing_id = get_briefing_id_from_date(briefing.briefing_date)
    for i, section in enumerate(briefing.sections, start=1):
        section_id = briefing_id * 100 + i
        rows.append(
            {
                "section_id": section_id,
                "briefing_id": briefing_id,
                "section_title": section.title,
                "section_text": section.text,
            }
        )

section_data = pd.DataFrame(rows)

section_data.info()
section_data.head(2)

100%|██████████| 2079/2079 [00:00<00:00, 8969.60it/s]


<class 'pandas.DataFrame'>
RangeIndex: 15087 entries, 0 to 15086
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   section_id     15087 non-null  int64
 1   briefing_id    15087 non-null  int64
 2   section_title  15087 non-null  str  
 3   section_text   15087 non-null  str  
dtypes: int64(2), str(2)
memory usage: 15.0 MB


,section_id,briefing_id,section_title,section_text
0,2021010401,20210104,PAKISTAN,"In a statement issued earlier today, the Secre..."
1,2021010402,20210104,NIGER,"In Niger, the Office for the Coordination of H..."


## 2. Check data integrity
Inspect the shortest and longest titles, to verify if they are valid and not artifacts of incorrect data processing.

In [5]:
section_data["title_length"] = section_data["section_title"].apply(lambda x: len(x))
section_data["text_length"] = section_data["section_text"].apply(lambda x: len(x))

section_data.info()
section_data.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 15087 entries, 0 to 15086
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   section_id     15087 non-null  int64
 1   briefing_id    15087 non-null  int64
 2   section_title  15087 non-null  str  
 3   section_text   15087 non-null  str  
 4   title_length   15087 non-null  int64
 5   text_length    15087 non-null  int64
dtypes: int64(4), str(2)
memory usage: 15.2 MB


,section_id,briefing_id,section_title,section_text,title_length,text_length
0,2021010401,20210104,PAKISTAN,"In a statement issued earlier today, the Secre...",8,440
1,2021010402,20210104,NIGER,"In Niger, the Office for the Coordination of H...",5,1357


In [6]:
section_data["title_length"].value_counts().sort_index()

title_length
2         3
3        30
4       557
5      2072
6       362
       ... 
113       1
116       1
121       1
123       1
129       1
Name: count, Length: 95, dtype: int64

In [7]:
briefing_ids = section_data.query(
    "title_length<=2 or title_length >= 120"
)[["briefing_id", "section_title"]]
briefing_ids.head(5)

for _, row in briefing_ids.iterrows():
    print(row["section_title"])
    date_string = str(row["briefing_id"])
    date_string = "-".join([date_string[:4], date_string[4:6], date_string[6:8]])
    url = f"{BASE_URL}/{date_string}.html"
    print(url)

G7
https://www.un.org/sg/en/content/highlight/2021-06-14.html
SECRETARY-GENERAL/DECLARATION ON THE RIGHTS OF PERSONS BELONGING TO NATIONAL OR ETHNIC, RELIGIOUS AND LINGUISTIC MINORITIES
https://www.un.org/sg/en/content/highlight/2022-09-21.html
G7
https://www.un.org/sg/en/content/highlight/2023-05-18.html
G7
https://www.un.org/sg/en/content/highlight/2024-06-12.html
CONCLUSION OF THE OPEN-ENDED WORKING GROUP ON SECURITY OF AND IN THE USE OF INFORMATION AND COMMUNICATIONS TECHNOLOGIES 2021-2025
https://www.un.org/sg/en/content/highlight/2025-07-14.html
GLOBAL INITIATIVE IN SUPPORT OF THE GLOBAL INITIATIVE TO GALVANIZE POLITICAL COMMITMENT TO INTERNATIONAL HUMANITARIAN LAW
https://www.un.org/sg/en/content/highlight/2025-09-22.html


## 3. Check frequency distribution of section titles
Sort the titles by frequency and decide a proper cutoff frequency to cover sufficient sections.

In [8]:
title_counts = section_data["section_title"].value_counts()

title_counts.info()
title_counts.head(2)

<class 'pandas.Series'>
Index: 3580 entries, OCCUPIED PALESTINIAN TERRITORY to STUDENTS/CLIMATE HAZARDS
Series name: count
Non-Null Count  Dtype
--------------  -----
3580 non-null   int64
dtypes: int64(1)
memory usage: 140.1 KB


section_title
OCCUPIED PALESTINIAN TERRITORY    576
UKRAINE                           554
Name: count, dtype: int64

In [9]:
## Calculate the number of titles and sections for different minimum counts of title occurrences
for min_count in range(1,11):
    kept = title_counts[title_counts >= min_count]

    print(
        f"min_count={min_count}: "
        f"{len(kept)} titles, "
        f"{kept.sum()} sections "
        f"({kept.sum() / title_counts.sum():.1%})"
    )

min_count=1: 3580 titles, 15087 sections (100.0%)
min_count=2: 992 titles, 12499 sections (82.8%)
min_count=3: 556 titles, 11627 sections (77.1%)
min_count=4: 397 titles, 11150 sections (73.9%)
min_count=5: 304 titles, 10778 sections (71.4%)
min_count=6: 255 titles, 10533 sections (69.8%)
min_count=7: 222 titles, 10335 sections (68.5%)
min_count=8: 200 titles, 10181 sections (67.5%)
min_count=9: 172 titles, 9957 sections (66.0%)
min_count=10: 154 titles, 9795 sections (64.9%)


In [10]:
## Choose a frequency cutoff for the titles to be included in the analysis
freq_cutoff = 7

major_title_counts = title_counts[title_counts >= freq_cutoff]

major_title_counts_text = "\n".join(
    f"{title}: {count}"
    for title, count in major_title_counts.items()
)

print(f"Total characters in titles text: {len(major_title_counts_text)}\n")
print(major_title_counts_text[:400])

Total characters in titles text: 4377

OCCUPIED PALESTINIAN TERRITORY: 576
UKRAINE: 554
SUDAN: 498
DEMOCRATIC REPUBLIC OF THE CONGO: 460
SECURITY COUNCIL: 357
YEMEN: 330
SYRIA: 313
SOUTH SUDAN: 313
FINANCIAL CONTRIBUTION: 308
HAITI: 289
AFGHANISTAN: 283
MYANMAR: 273
CENTRAL AFRICAN REPUBLIC: 271
LEBANON: 266
ETHIOPIA: 211
LIBYA: 181
MALI: 179
INTERNATIONAL DAYS: 170
SENIOR PERSONNEL APPOINTMENT: 137
DEPUTY SECRETARY-GENERAL: 128
SOMALI


## 4. Identify issues with LLM
Use the obtained `major_title_counts_text` (title + frequency) to extract the major recurring meaningful issues.

In [11]:
prompt = f"""\
**Task:**

Identify 15 to 25 major recurring issues from these UN Noon Briefing section titles and their frequencies:

{major_title_counts_text}

**Instructions:**

- Identify substantive issues; merge synonyms and titles referring to the same issue, but keep distinct issues separate.
- Use frequencies to prioritize recurring issues over rare topics.
- Keep prominent country- or territory-specific crises as standalone labels, rather than grouping them under broad categories such as 'Conflict and Crisis'.
- Exclude institutional, procedural, and administrative topics, including UN bodies, appointments, travel, briefing announcements, financial contributions, and international days.
- Use descriptive names. Avoid duplicate or strongly overlapping labels and issues not supported by the titles.

**Output Requirements:**

- Return only a list of issue names in the format: ['ISSUE A', 'ISSUE B', 'ISSUE C', ...]
- Do not include any explanation or additional text.
"""

In [12]:
llm = "llama3.2"

result = ollama.generate(
    model=llm,
    prompt=prompt,
    think=False,
)

print("issues:", result["response"])
print("\nmodel: ", llm)
print("created_at: ", result["created_at"])

issues: Here is the list of 23 major recurring issues based on the provided UN Noon Briefing section titles and their frequencies:

1. Palestine
2. Ukraine
3. Sudan
4. Democratic Republic of the Congo
5. Syria
6. South Sudan
7. Yemen
8. Lebanon
9. Afghanistan
10. Myanmar
11. Central African Republic
12. Somalia
13. Gaza
14. Libya
15. Mali
16. Colombia
17. Nigeria
18. Ethiopia
19. Iraq
20. Rohingya Refugees
21. Ebola
22. Children and Armed Conflict
23. COVID-19

Note: I excluded institutional, procedural, and administrative topics, as well as country-specific crises that are already represented by more specific labels (e.g., "Palestine" instead of "Occupied Palestinian Territory").

model:  llama3.2
created_at:  2026-10-04T19:23:10.739959944Z


## Simple checks on the LLM (Optional)

Perform some additional checks of the LLM parameters. 
- Token estimate: help adjust prompt length when needed.
- Duration and generation speed: to compare the performance of different LLMs.

In [13]:
## Token estimate
## For machine with VRAM <24 GB, Ollama default context length is 4096 tokens.

print("freq_cutoff: ", freq_cutoff)
print("number of titles: ", len(major_title_counts))
print("characters:", len(prompt))

estimated_tokens = round(len(prompt) / 2.5)
print("estimated tokens:", estimated_tokens)

freq_cutoff:  7
number of titles:  222
characters: 5324
estimated tokens: 2130


In [14]:
## Result evaluation

print("# Durations:")
print("total: {:.2f} s".format(result.total_duration / 1e9))
print("load: {:.2f} s".format(result.load_duration / 1e9))
print("prompt eval: {:.2f} s".format(result.prompt_eval_duration / 1e9))
print("generation: {:.2f} s".format(result.eval_duration / 1e9))

print("\n# Token Counts:")
print("prompt: {} tokens".format(result.prompt_eval_count))
print("output: {} tokens".format(result.eval_count))


prompt_speed = (
    result.prompt_eval_count
    / (result.prompt_eval_duration / 1e9)
)

generation_speed = (
    result.eval_count
    / (result.eval_duration / 1e9)
)

print("\n# Speeds:")
print(f"Prompt: {prompt_speed:.2f} tokens/s")
print(f"Generation: {generation_speed:.2f} tokens/s")

# Durations:
total: 73.33 s
load: 2.63 s
prompt eval: 0.71 s
generation: 69.89 s

# Token Counts:
prompt: 2253 tokens
output: 174 tokens

# Speeds:
Prompt: 3170.44 tokens/s
Generation: 2.49 tokens/s
